In [2]:
import pandas as pd
import numpy as np
#
#dataset
#
data={
    "weather":["sunny","sunny","cloudy","rainy","rainy","rainy","cloudy","sunny","sunny","rainy","sunny","cloudy","cloudy","rainy"],
    "temperature":["hot","hot","hot","mild","cool","cool","cool","mild","cold","mild","mild","mild","hot","mild"],
     "soil_moisture":["moist","dry","dry","dry","moist","moist","moist","dry","moist","moist","moist","dry","moist","dry"],
    "wind":["weak","strong","weak","weak","weak","strong","strong","weak","weak","weak","strong","strong","weak","strong"],
     "irrigate":["no","no","yes","yes","yes","no","yes","no","yes","yes","yes","yes","yes","no"]
}
df=pd.DataFrame(data)
print("==================================================================================================")
print(df)
print("==================================================================================================")

#step 1 entropy
#-------------------------------------------------------
def entropy(target):
  values,counts=np.unique(target,return_counts=True)
  prob=counts/counts.sum()
  return -np.sum(prob*np.log2(prob))

#------------------------------
#step 2 information gain

def information_gain(data,feature,target_name):
  total_entropy=entropy(data[target_name])
  values,counts=np.unique(data[feature],return_counts=True)

  weighted_entropy=0
  for i in range(len(values)):
    subset=data[data[feature]==values[i]]
    weighted_entropy+=(counts[i]/np.sum(counts))*entropy(subset[target_name])


  return total_entropy-weighted_entropy


#step 3 ID3 ALGORITHM

def id3(data,features,target_name,depth=0):
  indent=""*depth
  #if all target values ae same return leaf
  if len(features)==0:
    return data[target_name].mode()[0]
  #if no features left return majority class
  if len(features)==0:
    return data[target_name].mode()[0]

  #complete IG for all features
  gains=[]
  print(f"\n {indent}Evaluating features at depth {depth}")


  for feature in features:
    gain=information_gain(data,features,target_name)
    gains.append(gain)
    print(f"{indent} G ({feature})={gain:.4f}")

  #SELECT BEAT FEATURE
  best_feature=features[np.argmax(gains)]
  print(f"{indent} Splitting {best_feature}")
  tree={best_feature:{}}


  #split dataset

  for value in np.unique(data[best_feature]):
    print(f"{indent} splitting {best_feature}={value}")
    subset=data[data[best_feature]==value]
    if subset.shape[0]==0:
      tree[best_feature][value]=data[target_name].mode()[0]
    else:
      remaining_features=[f for f in features if f != best_feature]
      subtree=id3(subset,remaining_features,target_name,depth+1)
      tree[best_feature][value]=subtree

  return tree
#step 4:print tree function
def print_tree(tree,indent=""):
  if not isinstance(tree,dict):
    print(indent+"--->",tree)
    return


  for feature,branches in tree.items():
    for value,subtree in branches.items():
      print(f"{indent}{feature}={value}:")
      print_tree(subtree,indent+"")



#step 5 build tree
features=list(df.columns[:-1])
tree_model=id3(df,features,"irrigate")
#step 6 display tree

print("\n Final decision Tree:\n")
print_tree(tree_model)












   weather temperature soil_moisture    wind irrigate
0    sunny         hot         moist    weak       no
1    sunny         hot           dry  strong       no
2   cloudy         hot           dry    weak      yes
3    rainy        mild           dry    weak      yes
4    rainy        cool         moist    weak      yes
5    rainy        cool         moist  strong       no
6   cloudy        cool         moist  strong      yes
7    sunny        mild           dry    weak       no
8    sunny        cold         moist    weak      yes
9    rainy        mild         moist    weak      yes
10   sunny        mild         moist  strong      yes
11  cloudy        mild           dry  strong      yes
12  cloudy         hot         moist    weak      yes
13   rainy        mild           dry  strong       no

 Evaluating features at depth 0
 G (weather)=-2.8671
 G (temperature)=-2.8671
 G (soil_moisture)=-2.8671
 G (wind)=-2.8671
 Splitting weather
 splitting weather=cloudy

 Evaluating features